In [1]:
# Topic clustering notebook
from sklearn.metrics import silhouette_score
from pathlib import Path
import os
from collections import Counter

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

PROJECT_ROOT = Path("..").resolve()
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

print("Project root:", PROJECT_ROOT)
print("Processed data dir:", DATA_PROCESSED)


c:\Users\User\Documents\retailmind-prototype\.conda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: C:\Users\User\Documents\retailmind-prototype
Processed data dir: C:\Users\User\Documents\retailmind-prototype\data\processed


In [2]:
# Load the issue-tagged dataset produced by issue_tagging.ipynb (LLM-based)
tagged_path = DATA_PROCESSED / "uss_english_issue_tagged_llm_subset.parquet"
df = pd.read_parquet(tagged_path)

print("Total rows in tagged dataset:", len(df))
print("Columns:", df.columns.tolist())
df.head()



Total rows in tagged dataset: 39102
Columns: ['conv_id', 'turn_id', 'speaker', 'text', 'dialog_act', 'scores_raw', 'scores', 'is_overall', 'satisfaction_mean', 'label', 'dataset', 'low_satisfaction', 'entity_tag', 'issues', 'severity', 'reason', 'system_text_only', 'last_system_text', 'llm_satisfaction_score', 'llm_satisfaction_reason', 'low_satisfaction_llm']


,conv_id,turn_id,speaker,text,dialog_act,scores_raw,scores,is_overall,satisfaction_mean,label,...,low_satisfaction,entity_tag,issues,severity,reason,system_text_only,last_system_text,llm_satisfaction_score,llm_satisfaction_reason,low_satisfaction_llm
0,0,1,SYSTEM,Do you like movies like Thor?,None,None,None,False,NaN,ENTITY_NAME+MOVIE_OR_SERIES,...,False,ENTITY_NAME+MOVIE_OR_SERIES,[],NONE,,Do you like movies like Thor?,Do you like movies like Thor?,NaN,None,False
1,0,2,USER,"No, I don't like Thor.",None,"3,2,2","[3, 2, 2]",False,2.333333,ENTITY_NAME+MOVIE_OR_SERIES,...,True,ENTITY_NAME+MOVIE_OR_SERIES,[],NONE,,None,Do you like movies like Thor?,NaN,None,False
2,0,3,SYSTEM,Ok. What is it about this type of movie that y...,None,None,None,False,NaN,OTHER,...,False,OTHER,[],NONE,,Ok. What is it about this type of movie that y...,Ok. What is it about this type of movie that y...,NaN,None,False
3,0,4,USER,I don't like all the,None,"3,2,2","[3, 2, 2]",False,2.333333,OTHER,...,True,OTHER,[],NONE,,None,Ok. What is it about this type of movie that y...,NaN,None,False
4,0,5,USER,Like the I don't know. Like is it voice acting?,None,"3,2,2","[3, 2, 2]",False,2.333333,OTHER,...,True,OTHER,[],NONE,,None,Ok. What is it about this type of movie that y...,NaN,None,False


In [3]:
# Select turns to be clustered into topics
mask_target = (
    (df["speaker"] == "USER") &
    (df["low_satisfaction_llm"] == True) &
    (df["issues"].apply(len) > 0)
)

df_cluster = df.loc[mask_target].copy()

print("Rows selected for clustering:", len(df_cluster))
print("Speaker distribution:")
print(df_cluster["speaker"].value_counts())


Rows selected for clustering: 506
Speaker distribution:
speaker
USER    506
Name: count, dtype: int64


In [4]:
# Basic sanity checks before embeddings
assert len(df_cluster) > 0, "No rows selected for topic clustering!"

print("Sample issues:")
display(df_cluster[["issues","severity"]].head(5))

print("\nSample texts:")
display(df_cluster[["text"]].head(5))


Sample issues:


,issues,severity
91,"[MISSING_CONTEXT, TONE_ISSUE]",MEDIUM
351,[MISSING_CONTEXT],MEDIUM
464,"[MISSING_CONTEXT, TONE_ISSUE]",MEDIUM
489,[MISSING_CONTEXT],MEDIUM
637,[MISSING_CONTEXT],MEDIUM



Sample texts:


,text
91,"Oh, I dislike it. It's I don't know if it's of..."
351,"So, I would say I'm kind of in the middle on t..."
464,I found the movie to be just too over the top ...
489,It's just boring boring to me.
637,"Nope, haven't seen that."


In [5]:
# Text used for topic clustering
df_cluster["cluster_text"] = df_cluster["text"].astype(str)

print("Example cluster_text:")
display(df_cluster["cluster_text"].head())


Example cluster_text:


91     Oh, I dislike it. It's I don't know if it's of...
351    So, I would say I'm kind of in the middle on t...
464    I found the movie to be just too over the top ...
489                       It's just boring boring to me.
637                             Nope, haven't seen that.
Name: cluster_text, dtype: object

In [6]:
EMB_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMB_MODEL_NAME)

texts = df_cluster["cluster_text"].astype(str).tolist()

embeddings = embedder.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Embeddings shape:", embeddings.shape)


Batches: 100%|██████████| 8/8 [00:04<00:00,  1.86it/s]

Embeddings shape: (506, 384)


In [7]:
def pick_k_by_silhouette(X, k_min=4, k_max=10, random_state=42):
    scores = []
    for k in range(k_min, k_max + 1):
        km = KMeans(n_clusters=k, random_state=random_state, n_init="auto")
        labels = km.fit_predict(X)
        score = silhouette_score(X, labels)
        scores.append((k, score))
    return scores

k_scores = pick_k_by_silhouette(embeddings, k_min=4, k_max=10, random_state=42)
df_k = pd.DataFrame(k_scores, columns=["k", "silhouette"])
display(df_k.sort_values("silhouette", ascending=False).head(10))

best_k = int(df_k.sort_values("silhouette", ascending=False).iloc[0]["k"])
print("Best k by silhouette:", best_k)


,k,silhouette
0,4,0.071112
6,10,0.070802
5,9,0.069960
1,5,0.066566
3,7,0.063026
4,8,0.062126
2,6,0.058196


Best k by silhouette: 4


In [8]:
K = best_k  # or set manually, e.g. K = 10

kmeans = KMeans(n_clusters=K, random_state=42, n_init="auto")
topic_ids = kmeans.fit_predict(embeddings)

df_cluster = df_cluster.copy()
df_cluster["topic_id"] = topic_ids

print("Topic counts:")
display(df_cluster["topic_id"].value_counts().sort_index())


Topic counts:


topic_id
0    153
1    144
2    123
3     86
Name: count, dtype: int64

In [9]:
import numpy as np
import ast

def normalize_issues(val):
    # numpy array -> list
    if isinstance(val, np.ndarray):
        return [str(x) for x in val.tolist()]

    # list -> list[str]
    if isinstance(val, list):
        return [str(x) for x in val]

    # string representation of list -> list[str]
    if isinstance(val, str):
        s = val.strip()
        if s.startswith("[") and s.endswith("]"):
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, list):
                    return [str(x) for x in parsed]
            except Exception:
                pass
        return [s] if s else []

    # None/NaN/other
    return []

df_cluster["issues"] = df_cluster["issues"].apply(normalize_issues)

print("Issues types after normalization:")
print(df_cluster["issues"].apply(type).value_counts())
print("Example issues:", df_cluster["issues"].head(5).tolist())


Issues types after normalization:
issues
<class 'list'>    506
Name: count, dtype: int64
Example issues: [['MISSING_CONTEXT', 'TONE_ISSUE'], ['MISSING_CONTEXT'], ['MISSING_CONTEXT', 'TONE_ISSUE'], ['MISSING_CONTEXT'], ['MISSING_CONTEXT']]


In [10]:
from collections import Counter

def top_issue_labels(issue_lists, topn=3):
    c = Counter()
    for lst in issue_lists:
        if isinstance(lst, list):
            c.update(lst)
    return [x for x, _ in c.most_common(topn)]

topic_labels = []
for tid, g in df_cluster.groupby("topic_id"):
    top_issues = top_issue_labels(g["issues"].tolist(), topn=2)
    label = f"Topic {tid}: " + (" / ".join(top_issues) if top_issues else "General")
    topic_labels.append((tid, label))

df_topic_labels = pd.DataFrame(topic_labels, columns=["topic_id", "topic_label"])
df_cluster = df_cluster.merge(df_topic_labels, on="topic_id", how="left")

df_cluster[["topic_id","topic_label"]].drop_duplicates().sort_values("topic_id").head(10)


,topic_id,topic_label
18,0,Topic 0: MISSING_CONTEXT / UNSUPPORTED_INTENT
113,1,Topic 1: MISSING_CONTEXT / UNSUPPORTED_INTENT
0,2,Topic 2: MISSING_CONTEXT / TONE_ISSUE
4,3,Topic 3: MISSING_CONTEXT / UNSUPPORTED_INTENT


In [11]:
df_topics_rows = df_cluster[[
    "dataset","conv_id","turn_id",
    "topic_id","topic_label",
    "issues","severity","reason",
    "cluster_text"
]].copy()

print("df_topics_rows:", df_topics_rows.shape)
df_topics_rows.head()


df_topics_rows: (506, 9)


,dataset,conv_id,turn_id,topic_id,topic_label,issues,severity,reason,cluster_text
0,CCPE,2,33,2,Topic 2: MISSING_CONTEXT / TONE_ISSUE,"[MISSING_CONTEXT, TONE_ISSUE]",MEDIUM,The bot fails to acknowledge the user's strong...,"Oh, I dislike it. It's I don't know if it's of..."
1,CCPE,13,22,2,Topic 2: MISSING_CONTEXT / TONE_ISSUE,[MISSING_CONTEXT],MEDIUM,The bot's question about whether the user like...,"So, I would say I'm kind of in the middle on t..."
2,CCPE,18,10,2,Topic 2: MISSING_CONTEXT / TONE_ISSUE,"[MISSING_CONTEXT, TONE_ISSUE]",MEDIUM,The bot fails to acknowledge the user's negati...,I found the movie to be just too over the top ...
3,CCPE,19,16,2,Topic 2: MISSING_CONTEXT / TONE_ISSUE,[MISSING_CONTEXT],MEDIUM,The bot's question about what the user dislike...,It's just boring boring to me.
4,CCPE,25,19,3,Topic 3: MISSING_CONTEXT / UNSUPPORTED_INTENT,[MISSING_CONTEXT],MEDIUM,The bot fails to acknowledge the user's previo...,"Nope, haven't seen that."


In [12]:
print("Issues column dtype:", df_cluster["issues"].dtype)
print("Top 10 types in issues:")
print(df_cluster["issues"].apply(type).value_counts().head(10))

# Show a few non-empty issues rows (if any)
mask_nonempty = df_cluster["issues"].apply(lambda x: isinstance(x, (list, tuple, np.ndarray)) and len(x) > 0)
print("Rows with non-empty issues:", mask_nonempty.sum())

display(df_cluster.loc[mask_nonempty, ["issues", "severity", "reason", "text"]].head(10))


Issues column dtype: object
Top 10 types in issues:
issues
<class 'list'>    506
Name: count, dtype: int64
Rows with non-empty issues: 506


,issues,severity,reason,text
0,"[MISSING_CONTEXT, TONE_ISSUE]",MEDIUM,The bot fails to acknowledge the user's strong...,"Oh, I dislike it. It's I don't know if it's of..."
1,[MISSING_CONTEXT],MEDIUM,The bot's question about whether the user like...,"So, I would say I'm kind of in the middle on t..."
2,"[MISSING_CONTEXT, TONE_ISSUE]",MEDIUM,The bot fails to acknowledge the user's negati...,I found the movie to be just too over the top ...
3,[MISSING_CONTEXT],MEDIUM,The bot's question about what the user dislike...,It's just boring boring to me.
4,[MISSING_CONTEXT],MEDIUM,The bot fails to acknowledge the user's previo...,"Nope, haven't seen that."
5,[MISSING_CONTEXT],MEDIUM,The bot assumes the user is familiar with 'Chr...,Don't really know what that is.
6,"[MISSING_CONTEXT, TONE_ISSUE]",MEDIUM,The bot's question disregards the user's expre...,They annoy me. I like movies that have a theme...
7,[MISSING_CONTEXT],MEDIUM,The bot's response 'why?' does not acknowledge...,I just don't really care too much for the subj...
8,"[MISSING_CONTEXT, TONE_ISSUE]",HIGH,The bot misinterprets the user's feelings abou...,Not at all.
9,"[MISSING_CONTEXT, TONE_ISSUE]",HIGH,The bot's response lacks context and fails to ...,"Hello, how are you?"


In [13]:
def summarize_topic(group: pd.DataFrame, topn_examples=5):
    n = len(group)
    top_issues = top_issue_labels(group["issues"].tolist(), topn=5)

    # example texts: take first N (could be improved by centroid similarity; keep simple)
    example_texts = group["text"].astype(str).head(topn_examples).tolist()

    # reason: pick a representative reason if available
    example_reason = ""
    reasons = group["reason"].astype(str)
    reasons = [r for r in reasons.tolist() if r and r != "nan"]
    if reasons:
        example_reason = reasons[0]

    return pd.Series({
        "n_examples": n,
        "top_issues": top_issues,
        "example_texts": example_texts,
        "example_reason": example_reason,
    })

df_topics_summary = (
    df_cluster
    .groupby(["topic_id","topic_label"], as_index=False)
    .apply(lambda g: summarize_topic(g), include_groups=False)
)

print("df_topics_summary:", df_topics_summary.shape)
df_topics_summary.sort_values("n_examples", ascending=False).head(10)


df_topics_summary: (4, 6)


,topic_id,topic_label,n_examples,top_issues,example_texts,example_reason
0,0,Topic 0: MISSING_CONTEXT / UNSUPPORTED_INTENT,153,"[MISSING_CONTEXT, UNSUPPORTED_INTENT, WRONG_FA...","[I don't look for to own them anytime soon., I...",The bot fails to acknowledge the user's disint...
1,1,Topic 1: MISSING_CONTEXT / UNSUPPORTED_INTENT,144,"[MISSING_CONTEXT, UNSUPPORTED_INTENT, WRONG_FA...","[No, I'm sorry. I'll be checking in on the 3rd...",The bot incorrectly confirmed the check-in dat...
2,2,Topic 2: MISSING_CONTEXT / TONE_ISSUE,123,"[MISSING_CONTEXT, TONE_ISSUE, UNSUPPORTED_INTENT]","[Oh, I dislike it. It's I don't know if it's o...",The bot fails to acknowledge the user's strong...
3,3,Topic 3: MISSING_CONTEXT / UNSUPPORTED_INTENT,86,"[MISSING_CONTEXT, UNSUPPORTED_INTENT, TONE_ISS...","[Nope, haven't seen that., Don't really know w...",The bot fails to acknowledge the user's previo...


In [15]:
df_enriched = df.merge(
    df_topics_rows[["dataset","conv_id","turn_id","topic_id","topic_label"]],
    on=["dataset","conv_id","turn_id"],
    how="left",
)

df_enriched["topic_id"] = df_enriched["topic_id"].fillna(-1).astype(int)
df_enriched["topic_label"] = df_enriched["topic_label"].fillna("UNCLUSTERED")

print("Rows in df_enriched:", len(df_enriched))
df_enriched[["topic_id","topic_label"]].value_counts().head(10)


Rows in df_enriched: 39102


topic_id  topic_label                                  
-1        UNCLUSTERED                                      38596
 0        Topic 0: MISSING_CONTEXT / UNSUPPORTED_INTENT      153
 1        Topic 1: MISSING_CONTEXT / UNSUPPORTED_INTENT      144
 2        Topic 2: MISSING_CONTEXT / TONE_ISSUE              123
 3        Topic 3: MISSING_CONTEXT / UNSUPPORTED_INTENT       86
Name: count, dtype: int64

In [16]:
topics_rows_path = DATA_PROCESSED / "uss_english_topics_rows_llm_subset.parquet"
topics_summary_path = DATA_PROCESSED / "uss_english_topics_summary_llm_subset.parquet"
enriched_path = DATA_PROCESSED / "uss_english_turns_topics_llm_subset.parquet"

df_topics_rows.to_parquet(topics_rows_path, index=False)
df_topics_summary.to_parquet(topics_summary_path, index=False)
df_enriched.to_parquet(enriched_path, index=False)

print("Saved:")
print(" -", topics_rows_path)
print(" -", topics_summary_path)
print(" -", enriched_path)


Saved:
 - C:\Users\User\Documents\retailmind-prototype\data\processed\uss_english_topics_rows_llm_subset.parquet
 - C:\Users\User\Documents\retailmind-prototype\data\processed\uss_english_topics_summary_llm_subset.parquet
 - C:\Users\User\Documents\retailmind-prototype\data\processed\uss_english_turns_topics_llm_subset.parquet
